In [1]:
import re
import string
import pandas as pd
from collections import Counter

In [2]:
def normalize_abstract(raw_abstract: str):
    """Strip markup, collapse whitespace, lowercase, strip punctuation."""
    if not raw_abstract:
        return ""
 
    text = raw_abstract
 
    text = re.sub(r"<[^>]+>", " ", text)          # HTML/XML tags
    text = re.sub(r"\$.*?\$", " ", text)          # inline LaTeX math
    text = re.sub(r"\\[a-zA-Z]+\{[^}]*\}", " ", text)  # \command{...}
    text = re.sub(r"\\[a-zA-Z]+", " ", text)      # bare \command
 
    # Collapse whitespace / line breaks
    text = re.sub(r"\s+", " ", text).strip()
 
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
 
    # Collapse any double spaces left behind by punctuation removal
    text = re.sub(r"\s+", " ", text).strip()
    return text

GROUPS = {
    "capability": re.compile(r"^(software|comput|program)"),
    "domain":     re.compile(r"^quantum"),
    "artifact":   re.compile(r"^(framework|platform|librar|service|api|sdk)"),
}

In [3]:
df = pd.read_csv("un_matched_abstract_populated(manual_inspected).csv", dtype=str, encoding="latin-1").fillna("")
df["Abstract_Normalized"] = df["Abstract_Normalized"].map(normalize_abstract)
df["tokens"] = df["Abstract_Normalized"].map(lambda s: s.split())

In [4]:
def group_hits(tokens: str):
    """
    Return list of hit tokens' indexes (used for spaning)
    and list of hit records
    """
    hits = {g: [] for g in GROUPS}
    records = {g: Counter() for g in GROUPS}
    for i, tok in enumerate(tokens):
        for g, pat in GROUPS.items():
            if pat.match(tok):
                hits[g].append(i)
                records[g][tok] +=1
    return hits, records

In [5]:
def min_covering_span(hits: dict):
    if any(len(v) == 0 for v in hits.values()):
        return None
    
    merged = sorted((pos, g) for g, poslist in hits.items() for pos in poslist)
    need = len(GROUPS)
    count = Counter()
    have = 0
    best = None
    left = 0
    for _,(pos, g) in enumerate(merged):
        count[g] += 1
        if count[g] == 1: 
            have += 1
        while have == need: # slowly reduce the span between group
            span = pos - merged[left][0] + 1
            best = span if best is None else min(best, span)
            lg = merged[left][1]
            count[lg] -= 1
            if count[lg] == 0:
                have -= 1
            left += 1
    return best

In [6]:
def pair_dist(a, b):
    if not a or not b: return None
    return min(abs(i - j) for i in a for j in b)

In [7]:
rows = []
all_records = {g: Counter() for g in GROUPS}
for _, r in df.iterrows():
    hits, records = group_hits(r["tokens"])
    for g in GROUPS:
        all_records[g] += records[g]
    rows.append({
        "DOI": r["DOI"],
        "has_capability": bool(hits["capability"]),
        "has_domain":     bool(hits["domain"]),
        "has_artifact":   bool(hits["artifact"]),
        "boolean_match":  all(hits[g] for g in GROUPS),
        "d_cap_dom": pair_dist(hits["capability"], hits["domain"]),
        "d_dom_art": pair_dist(hits["domain"], hits["artifact"]),
        "d_cap_art": pair_dist(hits["capability"], hits["artifact"]),
        "min_span":  min_covering_span(hits),
    })
prox = pd.DataFrame(rows)

In [8]:
n = len(prox)
print("== boolean coverage (all 3 groups present) ==")
print(prox[["has_capability","has_domain","has_artifact","boolean_match"]].sum())
print(f"\nmatch all 3: {prox['boolean_match'].sum()}/{n}")

missed = prox[~prox["boolean_match"]]
if len(missed):
    print("\n-- abstracts missing a group (recall loss, unfixable by proximity) --")
    print(missed[["DOI","has_capability","has_domain","has_artifact"]].to_string(index=False))

spans = prox["min_span"].dropna()
# print("\n== min covering span (tokens) ==")
# print(spans.describe(percentiles=[.5,.75,.9,.95]).round(1))

print("\n== recall vs NEAR/W window ==")
for W in [3,4,5,6,7,8,10,15,20,30,50,100]:
    keep = (spans <= W).sum()
    print(f"NEAR/{W:<3}  keeps {keep:>2}/{n}  ({keep/n:.0%})")

== boolean coverage (all 3 groups present) ==
has_capability    84
has_domain        87
has_artifact      69
boolean_match     66
dtype: int64

match all 3: 66/87

-- abstracts missing a group (recall loss, unfixable by proximity) --
                            DOI  has_capability  has_domain  has_artifact
     10.1038/s41566-024-01403-4            True        True         False
       10.1088/2058-9565/ab9acb            True        True         False
        10.1145/3613424.3614300            True        True         False
   10.1007/978-3-030-21500-2_13            True        True         False
       10.1109/tqe.2023.3275868            True        True         False
        10.1145/3445814.3446750            True        True         False
    10.1007/978-3-031-64136-7_4            True        True         False
        10.1145/3665870.3665874            True        True         False
        10.1145/3659996.3660036           False        True          True
    10.1007/978-3-031-6413

In [9]:
PAIRS = {
    "d_cap_dom": "capability <-> domain",
    "d_dom_art": "domain     <-> artifact",
    "d_cap_art": "capability <-> artifact",
}

print("pairwise min token distance (over abstracts where both groups present)")
summary = prox[list(PAIRS)].describe().round(1)
print(summary)

print("\nBinding constraint between groups")
means = {p: prox[p].mean() for p in PAIRS}
for p, label in PAIRS.items():
    print(f"{label:24s} mean {means[p]:6.1f}  median {prox[p].median():6.1f}  max {prox[p].max():6.0f}")
widest = max(means, key=means.get)
print(f"\nwidest pair -> {PAIRS[widest]}: apply the NEAR window here; "
      f"the other pairs are usually adjacent collocations.")

# how often is each pair already tight (collocation, distance <= 2)?
print("\n== share adjacent (distance <= 2) ==")
for p, label in PAIRS.items():
    s = prox[p].dropna()
    print(f"{label:24s} {(s <= 2).sum():>2}/{len(s)}  ({(s <= 2).mean():.0%})")


pairwise min token distance (over abstracts where both groups present)
       d_cap_dom  d_dom_art  d_cap_art
count       84.0       69.0       66.0
mean         1.2        4.7        9.6
std          1.6        6.4       18.3
min          1.0        1.0        1.0
25%          1.0        2.0        1.0
50%          1.0        3.0        4.0
75%          1.0        5.0        8.0
max         15.0       47.0      124.0

Binding constraint between groups
capability <-> domain    mean    1.2  median    1.0  max     15
domain     <-> artifact  mean    4.7  median    3.0  max     47
capability <-> artifact  mean    9.6  median    4.0  max    124

widest pair -> capability <-> artifact: apply the NEAR window here; the other pairs are usually adjacent collocations.

== share adjacent (distance <= 2) ==
capability <-> domain    82/84  (98%)
domain     <-> artifact  33/69  (48%)
capability <-> artifact  25/66  (38%)
